# 初回投稿用分析：10-0 Run発生時の勝率

**目的**: NBA 2023-24シーズンで10-0の連続得点（Run）が起きた試合の勝率を分析する

**投稿テーマ**: 「10-0ランが起きた試合の勝率」

In [ ]:
# ライブラリインポート
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
from data_loader import NBADataLoader
from run_detector import RunDetector
from visualizer import BasketballVisualizer

print("ライブラリ読み込み完了")

## 1. データ取得

In [ ]:
# データローダー初期化
loader = NBADataLoader()

# 2023-24シーズンの全試合取得
print("NBA 2023-24シーズンデータ取得中...")
games = loader.get_season_games('2023-24', 'Regular Season')

print(f"\n取得完了: {len(games)}レコード")
print(f"ユニーク試合数: {len(games['GAME_ID'].unique())}試合")
print(f"\nカラム: {list(games.columns)}")

In [ ]:
# データの確認
games[['GAME_ID', 'GAME_DATE', 'TEAM_ABBREVIATION', 'MATCHUP', 'WL', 'PTS']].head(10)

## 2. Run検出（サンプル試合）

まずは少数の試合でテストします。

In [ ]:
# サンプル試合でテスト（最初の10試合）
sample_game_ids = games['GAME_ID'].unique()[:10]

detector = RunDetector()
all_runs = []

print("サンプル試合からRun検出中...")
for game_id in sample_game_ids:
    print(f"処理中: {game_id}")
    
    # プレイバイプレイ取得
    pbp = loader.get_play_by_play(game_id)
    
    if not pbp.empty:
        # Run検出
        runs = detector.detect_runs_from_game(pbp, run_sizes=[5, 8, 10])
        
        if not runs.empty:
            all_runs.append(runs)
            print(f"  → {len(runs)} Runs検出")

# 統合
if all_runs:
    all_runs_df = pd.concat(all_runs, ignore_index=True)
    print(f"\n合計Run検出数: {len(all_runs_df)}")
    print(all_runs_df.head())
else:
    print("Runが検出されませんでした")

## 3. 勝率分析（サンプル）

In [ ]:
# 10-0 Run発生時の勝率分析
if all_runs:
    result = detector.analyze_run_win_rate(games, all_runs_df, run_size=10)
    
    print("\n=== 10-0 Run発生時の勝率分析結果 ===")
    print(f"Runサイズ: {result['run_size']}-0")
    print(f"全試合数: {result['total_games']}試合")
    print(f"Run発生試合数: {result['games_with_run']}試合")
    print(f"Run未発生試合数: {result['games_without_run']}試合")
    print(f"\nRun発生時の勝率: {result['win_rate_with_run']:.1f}%")
    print(f"Run未発生時の勝率: {result['win_rate_without_run']:.1f}%")
    print(f"差分: +{result['difference']:.1f}%")

## 4. 可視化

In [ ]:
# グラフ作成
if all_runs:
    visualizer = BasketballVisualizer()
    
    # 勝率比較グラフ
    visualizer.create_win_rate_comparison(result)
    
    # 統計カード
    stat_value = f"{result['win_rate_with_run']:.1f}%"
    visualizer.create_simple_stat_card(stat_value, "10-0 Run発生時の勝率")
    
    print("\n可視化完了！outputs/images/ に保存されました")

## 5. 投稿文案作成

In [ ]:
if all_runs:
    # 投稿文テンプレート
    post_template = f"""【データ】
NBA 2023-24シーズンを分析すると、
「10-0の連続得点（Run）」が起きた試合の勝率は{result['win_rate_with_run']:.1f}%でした。

感覚的にはもっと高そうですが、
実際は{result['win_rate_with_run']:.1f}%に留まります。

【条件】
・NBA 2023-24レギュラーシーズン
・全{result['total_games']}試合を分析

※選手・チーム評価が目的ではなく、
現象としての分析です。

#NBA #データ分析 #Basketball
"""
    
    print("\n=== 投稿文案 ===")
    print(post_template)
    
    # ファイルに保存
    with open('../outputs/reports/first_post_draft.txt', 'w', encoding='utf-8') as f:
        f.write(post_template)
    
    print("\n投稿文案を outputs/reports/first_post_draft.txt に保存しました")

## 6. 次のステップ

- [ ] 全試合でRun検出を実行（時間がかかるため別スクリプト推奨）
- [ ] 5-0, 8-0ランの分析も実施
- [ ] クォーター別の分析
- [ ] 投稿スケジュールの作成

---

## メモ

**このNotebookの目的**:
- 初回投稿「10-0ランの勝率」のデータを取得・分析
- 投稿文と画像を作成

**注意点**:
- プレイバイプレイデータの取得にはAPI制限があるため、少しずつ実行
- 全1,230試合の分析には数時間かかる可能性あり
- キャッシュ機能を活用して再実行時の時間を短縮